# Yeoubi: decay branch

Finishes the main run into a usable model. It starts from the newest checkpoint in
`CKPT_REPO`, lowers the LR linearly from its peak to 10% over `DECAY_STEPS` steps on fresh
data, and writes everything to `DECAY_REPO`. `CKPT_REPO` is only read, so the main run
(`colab_train.ipynb` / `kaggle_train.ipynb`) can continue from where it stood, and a later
decay branch can be cut from a later checkpoint into a new `DECAY_REPO`.

- Runs on Colab (T4) or Kaggle (GPU T4 x2, Internet on). Needs an `HF_TOKEN` secret with
  write access, as the training notebooks do.
- One branch can span several sessions: each one resumes from the newest checkpoint in
  `DECAY_REPO`. The starting point is recorded in `DECAY_REPO/decay.json` on the first
  session, so later sessions use it even after `CKPT_REPO` has moved on.
- Done when `final.pt` appears in `DECAY_REPO`: that file is the decayed model.

In [ ]:
# SPDX-License-Identifier: BigCode-OpenRAIL-M-1.0
REPO = "https://github.com/Fury7425/Yeoubi.git"
HF_REPO = ""      # dataset repo holding the parts, such as "your-name/yeoubi-data"
CKPT_REPO = ""    # the main run's checkpoints, such as "your-name/yeoubi-ckpt"; only read
DECAY_REPO = ""   # this branch's own repo, such as "your-name/yeoubi-decay-44855"
DECAY_STEPS = 6000   # about 10-20% of the steps before it. Used on the first session only;
                     # later sessions read the value recorded in decay.json.
WINDOW = 24   # parts fetched per session, about 40 minutes each on one T4
KEEP = 2      # resumable checkpoints kept in DECAY_REPO, roughly 1 GB each
HOURS = 11    # train.py stops itself after this many hours and saves step_*_preempted.pt,
              # before Kaggle's 12 h limit. On Colab a disconnect can come sooner; the
              # every-500-steps checkpoints cover that.

In [ ]:
import glob
import gzip
import json
import os
import random
import re
import shutil
import threading
import time

assert HF_REPO and CKPT_REPO and DECAY_REPO, "set HF_REPO, CKPT_REPO and DECAY_REPO"
assert DECAY_REPO != CKPT_REPO, "the branch must not write into the main run's repo"

# /tmp on both hosts: Kaggle would copy anything under /kaggle/working out as notebook
# output at the end of a committed run.
WORK = "/tmp"
if not os.path.isdir(f"{WORK}/Yeoubi"):
    !git clone --depth 1 {REPO} {WORK}/Yeoubi
%cd {WORK}/Yeoubi
!pip -q install -r requirements.txt
!nvidia-smi -L

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
else:
    print("no HF_TOKEN secret: the checkpoint repo calls below will fail")

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

api = HfApi()
api.create_repo(DECAY_REPO, private=True, exist_ok=True)
LOCAL_CKPT = f"{WORK}/ckpt"
os.makedirs(LOCAL_CKPT, exist_ok=True)

In [ ]:
def step_of(path):
    match = re.search(r"step_(\d+)", os.path.basename(path))
    return int(match.group(1)) if match else -1


def is_resumable(name):
    return name.startswith("step_") and name.endswith(".pt") and "pre_a4" not in name


def newest_in(repo):
    return max(filter(is_resumable, api.list_repo_files(repo)), key=step_of, default="")


def gunzip(name, target, seed=None):
    # train.py's loader dispatches on .jsonl / .pt and does not read .gz itself.
    with gzip.open(hf_hub_download(HF_REPO, name, repo_type="dataset"), "rb") as source:
        lines = source.readlines()   # one part fits in RAM by design
    if seed is not None:
        random.Random(seed).shuffle(lines)   # same order as the training notebooks
    with open(target, "wb") as output:
        output.writelines(lines)


decay_files = api.list_repo_files(DECAY_REPO)
assert "final.pt" not in decay_files, f"this branch is finished: final.pt is in {DECAY_REPO}"
if "decay.json" in decay_files:
    with open(hf_hub_download(DECAY_REPO, "decay.json"), encoding="utf-8") as handle:
        plan = json.load(handle)
else:
    # First session: the branch starts from the main run's newest checkpoint. Recorded,
    # because by the next session CKPT_REPO may hold later ones.
    start_file = newest_in(CKPT_REPO)
    assert start_file, f"{CKPT_REPO} has no checkpoint to decay from"
    plan = {"from": start_file, "start": step_of(start_file), "steps": DECAY_STEPS}
    api.upload_file(
        path_or_fileobj=json.dumps(plan).encode(), path_in_repo="decay.json", repo_id=DECAY_REPO
    )
DECAY_START, DECAY_LEN = plan["start"], plan["steps"]
max_steps = DECAY_START + DECAY_LEN

# After a disconnect the branch resumes from its own newest checkpoint. Before its first
# one, it needs the main run's starting checkpoint, which the main run deletes once it
# has saved two newer ones: avoid running both at once until this branch has saved.
newest = newest_in(DECAY_REPO)
if newest:
    resume, step = hf_hub_download(DECAY_REPO, newest), step_of(newest)
else:
    resume, step = hf_hub_download(CKPT_REPO, plan["from"]), DECAY_START

with open(hf_hub_download(HF_REPO, "meta.json", repo_type="dataset"), encoding="utf-8") as handle:
    meta = json.load(handle)
parts, steps_per_part = meta["parts"], meta["steps_per_part"]
# Same step-to-part mapping as the main run, so the branch reads the parts after the
# ones the main run had reached: data the model has not seen yet.
first = (step // steps_per_part) % parts
window_parts = [(first + index) % parts for index in range(min(WINDOW, parts))]
snapshot_download(
    HF_REPO,
    repo_type="dataset",
    max_workers=8,
    allow_patterns=[f"parts/part_{part:05d}.jsonl.gz" for part in window_parts] + ["val.jsonl.gz"],
)
shutil.rmtree(f"{WORK}/train", ignore_errors=True)
os.makedirs(f"{WORK}/train")
os.makedirs(f"{WORK}/val", exist_ok=True)
for index, part in enumerate(window_parts):
    gunzip(f"parts/part_{part:05d}.jsonl.gz", f"{WORK}/train/{index:03d}.jsonl", seed=part)
gunzip("val.jsonl.gz", f"{WORK}/val/val.jsonl")
print(f"decay {DECAY_START}..{max_steps} from {plan['from']}; resume={newest or plan['from']} step={step}")

In [ ]:
_sizes = {}
_sync_lock = threading.Lock()   # the background loop and the final sync must not overlap


def sync_checkpoints(stable_only=True):
    """Upload finished local checkpoints to DECAY_REPO and drop old ones there."""
    with _sync_lock:
        uploaded = False
        for path in sorted(glob.glob(f"{LOCAL_CKPT}/*.pt"), key=step_of):
            size = os.path.getsize(path)
            # torch.save writes in place, so a file is only known complete once its
            # size has stopped changing between two polls.
            if stable_only and _sizes.get(path) != size:
                _sizes[path] = size
                continue
            api.upload_file(
                path_or_fileobj=path, path_in_repo=os.path.basename(path), repo_id=DECAY_REPO
            )
            os.remove(path)   # only after the upload committed
            uploaded = True
        if not uploaded:
            return
        resumable = sorted(filter(is_resumable, api.list_repo_files(DECAY_REPO)), key=step_of)
        for old in resumable[:-KEEP]:
            api.delete_file(old, DECAY_REPO)
        if len(resumable) > KEEP:
            # Deleted files keep counting toward storage until the history is squashed.
            api.super_squash_history(DECAY_REPO)
            # Squashing drops old files from the history, but the Hub keeps their stored
            # bytes, still counted toward the storage limit, until they are deleted by
            # hash: in the main run's repo, 89 of them (78 GB) filled the account.
            live = {f.lfs.sha256 for f in api.list_repo_tree(DECAY_REPO, expand=True)
                    if getattr(f, "lfs", None)}
            dead = [f for f in api.list_lfs_files(DECAY_REPO) if f.file_oid not in live]
            if dead:
                api.permanently_delete_lfs_files(DECAY_REPO, dead)


def _sync_loop():
    while True:
        time.sleep(60)
        try:
            sync_checkpoints()
        except Exception as error:   # a network hiccup must not end the syncing
            print("checkpoint sync failed, will retry:", error)


if "_sync_thread" not in globals():
    _sync_thread = threading.Thread(target=_sync_loop, daemon=True)
    _sync_thread.start()

In [ ]:
import torch

# Model and batch flags must match the main run's (see colab_train.ipynb for why each is
# set); only the schedule differs. 16 chunks per optimizer step in total, whatever the
# GPU count, as meta.json's step counts assume.
# --decay_start_step / --decay_steps: hold, then linear decay to 10% of peak LR.
# --a4_warmup_fraction 0: stay A8. With max_steps cut to the branch's end, the default
# 10% would switch to A4 in the middle of the decay.
GPUS = torch.cuda.device_count()
assert GPUS in (1, 2, 4), f"16 chunks per step do not split evenly over {GPUS} GPUs"
ACCUM = 4 // GPUS
!PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node {GPUS} train.py \
    --train_data "{WORK}/train/*.jsonl" --val_data "{WORK}/val/*.jsonl" \
    --output_dir {LOCAL_CKPT} --resume_from "{resume}" --max_steps {max_steps} \
    --decay_start_step {DECAY_START} --decay_steps {DECAY_LEN} --a4_warmup_fraction 0 \
    --max_seq_len 4096 --batch_size 4 --grad_accum_steps {ACCUM} \
    --hidden_dim 768 --n_heads 12 --ffn_dim 2048 \
    --bf16 false --compile_model false --gradient_checkpointing false \
    --save_every 500 --muon_weight_decay 0.005 --max_runtime_hours {HOURS} \
    --num_workers 2 --wandb_project ""

sync_checkpoints(stable_only=False)   # train.py has exited, so every file is complete